In [2]:
!pip install pandas numpy scikit-learn xgboost joblib ydata-profiling

In [3]:
import pandas as pd
import numpy as np
file_path = "Enterprise_Cyber_Kill_Chain_Dataset.csv" 
df = pd.read_csv(file_path)
print("Original Columns found in CSV:", df.columns.tolist())
target = 'Financial_Loss' if 'Financial_Loss' in df.columns else 'Financial_Loss_USD'
df[target] = pd.to_numeric(
    df[target].astype(str).str.replace(r'[\$,]', '', regex=True), 
    errors='coerce'
)
df = df.dropna(subset=[target]).reset_index(drop=True)
df = df[np.isfinite(df[target])].reset_index(drop=True)
print(f"Cleaned dataset shape: {df.shape}")
print(f"Target column '{target}' stats: Mean=${df[target].mean():,.2f}, Max=${df[target].max():,.2f}")

Original Columns found in CSV: ['Incident_ID', 'Timestamp', 'Industry', 'Country', 'Company_Size', 'Employee_Count', 'Attack_Vector', 'Threat_Actor', 'Firewall', 'MFA', 'EDR', 'IDS', 'Security_Training', 'Password_Policy', 'Patch_Age_Days', 'Open_Vulnerabilities', 'CVSS_Score', 'Internet_Exposed', 'Security_Audit_Score', 'Phishing_Click', 'Credential_Stolen', 'Privilege_Escalation', 'Lateral_Movement', 'Persistence', 'Data_Encrypted', 'Data_Exfiltration_GB', 'Attack_Success', 'Attack_Stage', 'Attack_Complexity', 'Detection_Time_Min', 'Response_Time_Min', 'Downtime_Hours', 'Records_Compromised', 'Financial_Loss_USD', 'Recovery_Cost_USD', 'Cyber_Risk_Score', 'Risk_Level', 'Incident_Severity', 'Hour', 'DayOfWeek', 'Month', 'Business_Hours', 'Weekend', 'Zero_Day', 'Compliance', 'Vendor_Count', 'ThirdParty_Risk', 'Insider_Risk', 'Security_Maturity', 'SOC_Team_Size']
Cleaned dataset shape: (95471, 50)
Target column 'Financial_Loss_USD' stats: Mean=$75,515.66, Max=$12,655,175.64


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# 1. Feature Engineering: Create Risk Ratios & Composite Indicators
df_eng = df.copy()

# Vulnerability-Patch Exposure Score
df_eng['Patch_Vuln_Risk'] = df_eng['Patch_Age_Days'] * df_eng['Open_Vulnerabilities']

# Threat Severity Factor
df_eng['CVSS_Exfil_Impact'] = df_eng['CVSS_Score'] * df_eng['Data_Exfiltration_GB']

# Security Shield Count (Sum of active security controls)
df_eng['Control_Shield_Count'] = (
    df_eng['Firewall'].astype(int) + 
    df_eng['MFA'].astype(int) + 
    df_eng['EDR'].astype(int) + 
    df_eng['Security_Training'].astype(int)
)
df_eng['Unprotected_Risk_Ratio'] = (df_eng['CVSS_Score'] * df_eng['Data_Exfiltration_GB']) / (df_eng['Control_Shield_Count'] + 1)

# List of all feature names after engineering
features = [
    'Company_Size', 'Patch_Age_Days', 'Open_Vulnerabilities', 
    'CVSS_Score', 'Security_Audit_Score', 'Data_Exfiltration_GB',
    'Firewall', 'MFA', 'EDR', 'Security_Training',
    'Patch_Vuln_Risk', 'CVSS_Exfil_Impact', 'Control_Shield_Count', 'Unprotected_Risk_Ratio'
]

features = [f for f in features if f in df_eng.columns]

X = df_eng[features]

# 2. Log Transformation on Target Variable to handle high skewness
y_raw = df_eng[target]
y_log = np.log1p(y_raw)  # log(1 + y) handles extreme financial loss values smoothly

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ]
)

# Train-Test Split (using log-transformed target)
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# Preprocess features
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print("Engineered dataset shape:", X_train_proc.shape)
print("Log transformation applied to target variable successfully.")

Engineered dataset shape: (76376, 15)
Log transformation applied to target variable successfully.


In [5]:
import pandas as pd

# Safe Fallback Import
try:
    from ydata_profiling import ProfileReport
except ImportError:
    try:
        from data_profiling import ProfileReport  # Support for updated package releases
    except ImportError:
        from pandas_profiling import ProfileReport

# Fallback column detection in case variables were cleared
target_col = 'Financial_Loss' if 'Financial_Loss' in df.columns else 'Financial_Loss_USD'
available_cols = [c for c in df.columns if c != target_col]

print("Generating Data Profiling Report...")

# Minimal=True prevents memory/UI rendering crashes on large correlation matrices
profile = ProfileReport(
    df[available_cols + [target_col]], 
    title="SIH26105 - Telemetry & Cyber Risk Profiling", 
    minimal=False,
    explorative=True
)

# Export directly to file
profile.to_file("cyber_risk_profiling_report.html")
print("Profiling report created: 'cyber_risk_profiling_report.html'")
print("Open this file in your web browser to view full statistics and correlations.")

C:\Users\haren\AppData\Local\Temp\ipykernel_19436\1075650694.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Generating Data Profiling Report...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:07<00:00,  7.04it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profiling report created: 'cyber_risk_profiling_report.html'
Open this file in your web browser to view full statistics and correlations.


In [6]:
import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# 1. Tuned XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42
)

# 2. Gradient Boosting Regressor (Native scikit-learn substitute for LightGBM)
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    random_state=42
)

# 3. Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42
)

# Ensemble: Voting Regressor combining models
ensemble_model = VotingRegressor(
    estimators=[
        ('xgb', xgb_model),
        ('gb', gb_model),
        ('rf', rf_model)
    ]
)

# Train ensemble on log-transformed target
ensemble_model.fit(X_train_proc, y_train_log)

# Predict log values and transform back to original USD domain
y_pred_log = ensemble_model.predict(X_test_proc)
y_pred_usd = np.expm1(y_pred_log)
y_test_usd = np.expm1(y_test_log)

# Clip negative values
y_pred_usd = np.clip(y_pred_usd, a_min=0, a_max=None)

# Evaluate on original USD scale
mae = mean_absolute_error(y_test_usd, y_pred_usd)
r2 = r2_score(y_test_usd, y_pred_usd)

print("=== IMPROVED MODEL PERFORMANCE METRICS ===")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"R2 Score: {r2:.4f}")

=== IMPROVED MODEL PERFORMANCE METRICS ===
Mean Absolute Error (MAE): $50,036.27
R2 Score: 0.0764


In [7]:
import joblib

# Load your uncompressed model (if already in memory, skip this line)
# model = joblib.load("cyberloss.pkl")

# Save with Zlib/Gzip compression (level 3 provides the best speed-to-compression ratio)
joblib.dump(ensemble_model, "cyber_loss_model.pkl", compress=3)

print("Compressed model saved successfully!")

Compressed model saved successfully!


In [8]:
# In main.py
model = joblib.load("cyber_loss_model.pkl")  # Loads compressed joblib models seamlessly
preprocessor = joblib.load("preprocessor.pkl")

In [9]:
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import numpy as np
import joblib
import uvicorn

app = FastAPI(
    title="SIH26105 API", 
    description="Cyber Risk Quantification Engine (Log-Scaled Ensemble)"
)

# Load trained artifacts
model = joblib.load("cyber_loss_model.pkl")
preprocessor = joblib.load("preprocessor.pkl")

class TelemetryInput(BaseModel):
    Company_Size: str = "Medium"
    Patch_Age_Days: int = 45
    Open_Vulnerabilities: int = 25
    CVSS_Score: float = 7.2
    Security_Audit_Score: float = 60.0
    Data_Exfiltration_GB: float = 10.0
    Firewall: int = 1
    MFA: int = 0
    EDR: int = 0
    Security_Training: int = 0

class OptimizeInput(BaseModel):
    telemetry: TelemetryInput
    budget_usd: float

def compute_engineered_features(data_dict: dict) -> pd.DataFrame:
    df_in = pd.DataFrame([data_dict])
    
    # Feature engineering matching notebook logic
    df_in['Patch_Vuln_Risk'] = df_in['Patch_Age_Days'] * df_in['Open_Vulnerabilities']
    df_in['CVSS_Exfil_Impact'] = df_in['CVSS_Score'] * df_in['Data_Exfiltration_GB']
    df_in['Control_Shield_Count'] = (
        df_in['Firewall'].astype(int) + 
        df_in['MFA'].astype(int) + 
        df_in['EDR'].astype(int) + 
        df_in['Security_Training'].astype(int)
    )
    df_in['Unprotected_Risk_Ratio'] = (df_in['CVSS_Score'] * df_in['Data_Exfiltration_GB']) / (df_in['Control_Shield_Count'] + 1)
    
    return df_in

def get_financial_loss_prediction(df_in: pd.DataFrame) -> float:
    X_proc = preprocessor.transform(df_in)
    pred_log = model.predict(X_proc)[0]
    pred_usd = float(np.expm1(pred_log))
    return max(0.0, pred_usd)

@app.get("/")
def root():
    return {"status": "Online", "service": "SIH26105 Risk Engine"}

@app.post("/predict_risk")
def predict_risk(data: TelemetryInput):
    df_in = compute_engineered_features(data.dict())
    pred_loss = get_financial_loss_prediction(df_in)
    
    tier = "CRITICAL" if pred_loss > 500000 else "HIGH" if pred_loss > 200000 else "MEDIUM"
    return {"predicted_financial_loss_usd": round(pred_loss, 2), "risk_tier": tier}

@app.post("/optimize_investment")
def optimize_investment(data: OptimizeInput):
    base_dict = data.telemetry.dict()
    budget = data.budget_usd
    
    base_df = compute_engineered_features(base_dict)
    base_loss = get_financial_loss_prediction(base_df)
    
    controls = {
        "Enable MFA": {"field": "MFA", "cost": 15000},
        "Deploy EDR": {"field": "EDR", "cost": 30000},
        "Security Training": {"field": "Security_Training", "cost": 10000}
    }
    
    allocations = []
    rem_budget = budget
    current_loss = base_loss
    
    for name, item in controls.items():
        if base_dict[item["field"]] == 0 and rem_budget >= item["cost"]:
            temp_dict = base_dict.copy()
            temp_dict[item["field"]] = 1
            
            temp_df = compute_engineered_features(temp_dict)
            new_loss = get_financial_loss_prediction(temp_df)
            
            savings = current_loss - new_loss
            if savings > 0:
                allocations.append({
                    "control": name,
                    "cost_usd": item["cost"],
                    "risk_reduction_usd": round(savings, 2)
                })
                rem_budget -= item["cost"]
                base_dict[item["field"]] = 1
                current_loss = new_loss
                
    return {
        "baseline_loss_usd": round(base_loss, 2),
        "post_investment_loss_usd": round(current_loss, 2),
        "total_risk_reduced_usd": round(base_loss - current_loss, 2),
        "remaining_budget_usd": rem_budget,
        "recommendations": allocations
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

Overwriting main.py


In [10]:
!pip install fastapi uvicorn requests streamlit

In [11]:
import subprocess

# Start FastAPI server in background
process = subprocess.Popen(["python", "main.py"])
print(" FastAPI server launched! Access documentation at http://localhost:8000/docs")

 FastAPI server launched! Access documentation at http://localhost:8000/docs
